# Notebook 01_Data_Extraction_Preprocessing

## Introducción:
Más allá de la extracción, este notebook construye un **dataset robusto de alta fidelidad** que permite estudiar la transmisión de la política monetaria (FED) hacia los activos de riesgo (NASDAQ) y su posterior impacto en el ecosistema cripto (Bitcoin).

## Para este estudio, nos centraremos en tres activos:
1. **Bitcoin (BTC):** Activo de referencia en el ecosistema cripto.
2. **NASDAQ (^IXIC):** Índice que representa el sector tecnológico estadounidense, históricamente correlacionado con el apetito por el riesgo.
3. **FED Funds Rate:** Tasa de interés de referencia de la Reserva Federal, utilizada como indicador de la liquidez global y el costo del capital.

## 1. Configuración del Entorno y Modularización del código
Para mantener un código limpio, legible y modular, he separado la lógica económica (`/notebooks`) de la lógica técnica (`/src`). En esta sección, importamos los módulos personalizados que gestionan la conexión con las APIs de Yahoo Finance y FRED.

In [1]:
import sys
import os
from dotenv import load_dotenv

sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_extraction_processing import (
    download_market_data, 
    get_fred_data, 
    calculate_volatility, 
    clean_and_resample
)

print("✅ Funciones importadas correctamente desde src")

✅ Funciones importadas correctamente desde src


## 2. Definición de Parámetros y Seguridad
Utilizamos un archivo `.env` para gestionar la API Key de la FRED (privada) evitando así leakage de datos sensibles. Definimos el horizonte temporal de 10 años.

## Justificación del Periodo (2015 - 2025)
He seleccionado una ventana temporal de 10 años por dos razones críticas:
* **Madurez Institucional:** Antes de 2015, el Bitcoin carecía de la liquidez y la infraestructura necesaria para ser comparado seriamente con los mercados de renta variable, a pesar de que su origen se remonte a inicios de 2009.
* **Ciclos Económicos:** Este periodo cubre multitud de escenarios, desde la fase de tipos de interés cero, pasando por el choque del COVID-19, hasta el ciclo inflacionario (post-Covid) y la subida de tipos de 2023-2024.

In [2]:
load_dotenv() 

api_key = os.getenv('FRED_API_KEY')

if api_key:
    print("✅ API Key cargada exitosamente.")
else:
    print("❌ Error: No se encontró la API Key. Revisa tu archivo .env")

tickers = {'BTC-USD': 'btc', '^IXIC': 'nasdaq'}
start = '2015-01-01'
end = '2025-02-28'

✅ API Key cargada exitosamente.


## 3. Pipeline de extracción y resampling
En este paso, realizamos la descarga de datos vía API y aplicamos las transformaciones necesarias para construir un dataset de alta fidelidad:

1. **Cálculo de Volatilidad**: Estimamos la volatilidad del Bitcoin a partir de sus retornos diarios antes de agrupar los datos, preservando la información de riesgo intra-mes.
2. **Resampling Mensual (Month Start - MS)**: Transformamos la frecuencia de diaria a mensual. 
    * **Justificación Técnica**: Esta elección es estratégica para reducir el **ruido estocástico** y la volatilidad extrema de corto plazo inherente al Bitcoin. Al suavizar la serie, facilitamos que los modelos avanzados (como la Cointegración en el **Notebook_05**) identifiquen **tendencias estructurales** y relaciones de equilibrio de largo plazo que quedarían ocultas en datos diarios.
    * **Alineación Macro**: Sincronizamos los activos con la frecuencia de publicación de indicadores de la FED, permitiendo un análisis comparativo riguroso.
3. **Integridad de Datos**: Dado que el NASDAQ cierra fines de semana y festivos, aplicamos *Forward Fill* (`ffill`) para garantizar que cada observación de Bitcoin tenga su correspondiente métrica de mercado tradicional.

## Fuentes
Utilizaremos dos fuentes de datos de destacan por su alta fiabilidad:
* **Yahoo Finance (vía `yfinance`):** Para obtener los cierres diarios ajustados de BTC y el NASDAQ.
* **FRED (Federal Reserve Economic Data):** Para obtener la Tasa de Fondos Federales de forma directa de la base de datos de la Reserva Federal a traves de la API oficial.

La integración se realiza mediante módulos personalizados en la carpeta `src` (disponible en el repositorio GitHub), garantizando que el proceso sea reproducible y reciclable.

In [3]:
tickers = {'BTC-USD': 'btc', '^IXIC': 'nasdaq'}
market_raw = download_market_data(tickers, start, end)
fed_raw = get_fred_data('FEDFUNDS', api_key, start, end)

market_raw['btc_vol'] = calculate_volatility(market_raw)
df_final = clean_and_resample(market_raw, fed_raw)

# Guardamos en la carpeta data que creamos
df_final.to_csv('../data/bitcoin_nasdaq_extended_py.csv')
print("✅ Datos guardados en /data")

[*********************100%***********************]  2 of 2 completed


✅ Datos guardados en /data


In [6]:
# Validación rápida de la estructura final
print(f"Dimensiones del dataset: {df_final.shape}")
print(f"Valores nulos totales: {df_final.isnull().sum().sum()}")
display(df_final.head())
display(df_final.describe().T)

Dimensiones del dataset: (122, 4)
Valores nulos totales: 0


,btc,nasdaq,btc_vol,fed_rate
Date,,,,
2015-01-31,217.464005,4635.240234,1.482538,0.11
2015-02-28,254.263000,4963.529785,0.727239,0.11
2015-03-31,244.223999,4900.879883,0.644011,0.11
2015-04-30,236.145004,4941.419922,0.463459,0.12
2015-05-31,230.190002,5070.029785,0.266667,0.12


,count,mean,std,min,25%,50%,75%,max
btc,122.0,21650.430145,24254.856339,217.464005,3020.953247,10153.363770,34947.572266,102405.023438
nasdaq,122.0,10045.774730,4277.363952,4557.950195,6368.255127,8777.509766,13623.272705,19627.439453
btc_vol,122.0,0.628440,0.300823,0.217206,0.429855,0.559431,0.787388,1.964881
fed_rate,122.0,1.824344,1.885949,0.050000,0.130000,1.160000,2.417500,5.330000


## Conclusión de la Extracción y Proximos Pasos
Los datos han sido validados y guardados en la carpeta `/data`. Contamos con una muestra de aproximadamente 122 observaciones mensuales, libre de valores nulos y lista para el análisis descriptivo.

**Siguiente paso:** Proceder al **Notebook_02** para realizar el Análisis Exploratorio de Datos (EDA).

> **💾 Nota sobre el Almacenamiento**: 
> El dataset final se exporta en formato **CSV** a la carpeta `/data`. Se ha elegido este formato por su portabilidad y compatibilidad universal entre diferentes herramientas de análisis, asegurando que el estado de la investigación sea fácilmente reproducible en cualquier entorno.